In [ ]:
import logging
import os
import time
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import torch
from ase.io import read as ase_read

from nvalchemi.data import AtomicData, Batch
from nvalchemi.dynamics import initialize_velocities
from nvalchemi.dynamics.hooks import LoggingHook, ProfilerHook
from nvalchemi.dynamics.integrators.nvt_langevin import NVTLangevin
from nvalchemi.models.aimnet2 import AIMNet2Wrapper
from nvalchemi.models.ewald import EwaldModelWrapper
from nvalchemi.models.pipeline import PipelineGroup, PipelineModelWrapper
from nvalchemiops.torch.interactions.electrostatics.parameters import estimate_ewald_parameters

torch._functorch.config.donated_buffer = False
torch.set_float32_matmul_precision("high")
logging.basicConfig(level=logging.INFO)


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DT = 1.5                          # fs
FRICTION = 0.001                   # fs^-1 (= 1.0 ps^-1)
T_EQUIL = 200.0                    # K
LOG_EVERY = 100                    # steps

# Benchmark sweep: (n, 2n, n) supercell pattern
SUPERCELL_SCALES = [1, 2, 3, 4, 5]  # 72 -> 9000 atoms
BENCHMARK_PS = 0.5                    # NVT duration per size

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_DIR = f"logs/bench_{RUN_ID}"
os.makedirs(LOG_DIR, exist_ok=True)
print(f"Benchmark run {RUN_ID} -> {LOG_DIR}, device={DEVICE}")


In [ ]:
from helpers import compute_density, make_safety_hooks, stdout_writer


In [ ]:
unit_cell = ase_read("data/naphthalene.cif")
atoms_per_mol = len(unit_cell) // 2  # Z=2 for naphthalene P2_1/a
print(f"Unit cell: {len(unit_cell)} atoms, {atoms_per_mol} atoms/molecule")

In [ ]:
aimnet2 = AIMNet2Wrapper.from_checkpoint("aimnet2", device=DEVICE, compile_model=True)
print(f"AIMNet2 loaded on {DEVICE}, cutoff={aimnet2.model_config.neighbor_config.cutoff} A")

In [ ]:
bench_results = []
n_steps = int(BENCHMARK_PS * 1000 / DT)
print(f"Benchmark: {BENCHMARK_PS} ps = {n_steps} steps per size
")

for s in SUPERCELL_SCALES:
    sc = (s, 2 * s, s)
    supercell = unit_cell * sc
    n_atoms = len(supercell)
    n_mol = n_atoms // atoms_per_mol
    print(f"{'='*60}")
    print(f"Scale {s}: supercell {sc}, {n_atoms} atoms ({n_mol} molecules)")

    # Build AtomicData -> Batch
    n = len(supercell)
    data = AtomicData(
        positions=torch.tensor(supercell.get_positions(), dtype=torch.float32, device=DEVICE),
        atomic_numbers=torch.tensor(supercell.get_atomic_numbers(), dtype=torch.long, device=DEVICE),
        forces=torch.zeros(n, 3, device=DEVICE),
        energy=torch.zeros(1, 1, device=DEVICE),
        stress=torch.zeros(1, 3, 3, device=DEVICE),
        cell=torch.tensor(supercell.cell.array, dtype=torch.float32, device=DEVICE).unsqueeze(0),
        pbc=torch.tensor([[True, True, True]], device=DEVICE),
    )
    batch = Batch.from_data_list([data], device=DEVICE)
    print(f"  Cell lengths: {[f'{l:.2f}' for l in batch.cell.squeeze().norm(dim=-1).tolist()]} A")
    print(f"  Density: {compute_density(batch):.3f} g/cm3")

    # Estimate Ewald parameters and build pipeline for this cell size
    params = estimate_ewald_parameters(batch.positions, batch.cell, batch.batch_idx)
    ewald_cutoff = params.real_space_cutoff.max().item()
    ewald = EwaldModelWrapper(cutoff=ewald_cutoff, accuracy=1e-6, hybrid_forces=False)
    pipe = PipelineModelWrapper(
        groups=[PipelineGroup(steps=[aimnet2, ewald], use_autograd=True)]
    )
    pipe.set_config("active_outputs", {"energy", "forces", "stress", "charges"})
    print(f"  Ewald cutoff: {ewald_cutoff:.2f} A")

    # Initialize velocities
    batch.velocities = torch.zeros_like(batch.positions)
    initialize_velocities(
        batch.velocities, batch.atomic_masses,
        temperature=torch.tensor([T_EQUIL], device=DEVICE),
        batch_idx=batch.batch_idx, random_seed=42,
        remove_com=True, rescale=True,
    )

    # Set up NVT with profiler
    profiler = ProfilerHook("detailed", timer_backend="auto", frequency=1)
    nvt = NVTLangevin(
        model=pipe, dt=DT, temperature=T_EQUIL, friction=FRICTION,
        n_steps=n_steps,
        hooks=make_safety_hooks(pipe) + [profiler],
    )

    with LoggingHook(backend="custom", writer_fn=stdout_writer, frequency=LOG_EVERY) as out_log:
        nvt.register_hook(out_log)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        batch = nvt.run(batch)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0

    profiler.close()
    summary = profiler.summary()
    steps_per_sec = n_steps / elapsed
    ns_per_day = (BENCHMARK_PS / 1000) / (elapsed / 86400)

    bench_results.append({
        "scale": s,
        "supercell": sc,
        "n_atoms": n_atoms,
        "n_mol": n_mol,
        "elapsed_s": elapsed,
        "steps_per_sec": steps_per_sec,
        "ns_per_day": ns_per_day,
        "profiler": summary,
    })
    print(f"  Elapsed: {elapsed:.1f} s | {steps_per_sec:.1f} steps/s | {ns_per_day:.4f} ns/day")

    # Free GPU memory before next size
    del batch, data, pipe, ewald, nvt, profiler, params
    torch.cuda.empty_cache()

print(f"
Benchmark complete: {len(bench_results)} sizes tested")

In [ ]:
n_atoms_list = [r["n_atoms"] for r in bench_results]
elapsed_list = [r["elapsed_s"] for r in bench_results]
nsday_list = [r["ns_per_day"] for r in bench_results]

# Extract per-stage mean times for stacked bar chart
stage_names = []
stage_data = {}
for r in bench_results:
    for transition, stats in r["profiler"].items():
        if transition not in stage_data:
            stage_data[transition] = []
            stage_names.append(transition)
        stage_data[transition].append(stats["mean_s"] * 1000)  # ms

production_atoms = 4608  # (4,8,4) supercell

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: wall-clock time
ax = axes[0]
ax.plot(n_atoms_list, elapsed_list, "o-", markersize=8, color="steelblue")
ax.axvline(production_atoms, color="gray", ls="--", alpha=0.6, label=f"Production ({production_atoms})")
ax.set_xlabel("Atom count")
ax.set_ylabel("Wall-clock time (s)")
ax.set_title(f"NVT {BENCHMARK_PS} ps total runtime")
ax.legend()
ax.grid(True, alpha=0.3)

# Center: ns/day
ax = axes[1]
ax.plot(n_atoms_list, nsday_list, "s-", markersize=8, color="darkorange")
ax.axvline(production_atoms, color="gray", ls="--", alpha=0.6, label=f"Production ({production_atoms})")
ax.set_xlabel("Atom count")
ax.set_ylabel("ns/day")
ax.set_title("MD throughput")
ax.legend()
ax.grid(True, alpha=0.3)

# Right: per-stage breakdown (stacked bar)
ax = axes[2]
x = np.arange(len(n_atoms_list))
width = 0.6
bottom = np.zeros(len(n_atoms_list))
colors = plt.cm.Set2(np.linspace(0, 1, len(stage_names)))
for i, (name, color) in enumerate(zip(stage_names, colors)):
    vals = np.array(stage_data[name])
    short_name = name.split("->")[1] if "->" in name else name
    ax.bar(x, vals, width, bottom=bottom, label=short_name, color=color)
    bottom += vals
ax.set_xticks(x)
ax.set_xticklabels([str(n) for n in n_atoms_list], rotation=45)
ax.set_xlabel("Atom count")
ax.set_ylabel("Mean step time (ms)")
ax.set_title("Per-stage breakdown")
ax.legend(fontsize=7, loc="upper left")
ax.grid(True, alpha=0.3, axis="y")

fig.suptitle(f"Supercell Size Benchmark — AIMNet2 + Ewald NVT @ {T_EQUIL} K", fontsize=13)
plt.tight_layout()
plt.savefig(f"{LOG_DIR}/benchmark_supercell.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Summary table
header = f"{' Scale':>6} {'Supercell':>12} {'Atoms':>7} {'Elapsed (s)':>12} {'Steps/s':>9} {'ns/day':>10}"
print(header)
print("-" * len(header))
for r in bench_results:
    sc_str = f"({r['supercell'][0]},{r['supercell'][1]},{r['supercell'][2]})"
    print(f"{r['scale']:>6d} {sc_str:>12} {r['n_atoms']:>7d} {r['elapsed_s']:>12.1f} {r['steps_per_sec']:>9.1f} {r['ns_per_day']:>10.4f}")

print(f"
Per-stage timing breakdown (mean ms/step):")
for r in bench_results:
    print(f"
  Scale {r['scale']} ({r['n_atoms']} atoms):")
    for transition, stats in r["profiler"].items():
        print(f"    {transition}: {stats['mean_s']*1000:.2f} ms (std={stats['std_s']*1000:.2f})")